# Story 6: Pay for an Expensive Setup Only Once
**Who:** A researcher who evaluates a machine-learned interatomic potential for thousands of structures. Loading the
potential takes much longer than a single evaluation.

**What they want:** Load the potential once per worker and then stream many small evaluations through these workers.
If a worker crashes, the remaining work should continue.

**Why plain SLURM is painful:** Every job or job step starts a fresh Python process, so the potential is loaded again
for every structure - and the setup dominates the run time. Keeping a pool of long-lived worker processes alive in an
allocation and feeding them work means writing a custom server, and restarting crashed workers is left to you.

**How executorlib helps:** With `block_allocation=True` the `SlurmJobExecutor` starts `max_workers` long-lived workers
in the allocation. The `init_function` runs once per worker and its return values are passed as keyword arguments to
the submitted functions. `restart_limit` restarts crashed workers.

This notebook has to run **inside a SLURM allocation** with at least two cores, for example started with
`salloc -n 2`.

## Cluster-specific settings
**Adapt the following cell to your cluster and allocation.** The values below are the ones used to test this notebook in a two-core allocation of a single-node SLURM test cluster with MPICH. For OpenMPI >= 5 use `["--mpi=pmix"]`; set `max_cores` to the number of cores of your allocation.

In [ ]:
# Cluster-specific settings - adapt these values to your cluster and allocation.
slurm_cmd_args = ["--mpi=pmi2"]  # additional arguments for srun, e.g. the MPI plugin
max_cores = 2  # number of CPU cores of the allocation executorlib may use

## Step 1: The functions
`load_potential()` stands in for the expensive setup, `evaluate()` for the cheap evaluation which needs the loaded
potential.

In [ ]:
import time


def load_potential():
    # Stand-in for loading a large machine-learned potential from disk (slow).
    time.sleep(3)
    return {"potential": {"epsilon": 0.0104, "sigma": 3.4}}


def evaluate(distance, potential):
    # Stand-in for a cheap evaluation which requires the loaded potential.
    eps, sigma = potential["epsilon"], potential["sigma"]
    return 4 * eps * ((sigma / distance) ** 12 - (sigma / distance) ** 6)


distance_lst = [3.4, 3.6, 3.8, 4.0, 4.2, 4.4]

## Step 2: The naive way - setup in every call
Without block allocation every call is a separate `srun` job step with a fresh Python process, so every call has to
load the potential itself:

In [ ]:
from executorlib import SlurmJobExecutor


def evaluate_with_setup(distance):
    return evaluate(distance, **load_potential())


start = time.time()
with SlurmJobExecutor(max_cores=max_cores, resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args}) as exe:
    energy_lst = list(exe.map(evaluate_with_setup, distance_lst))
print(f"{len(energy_lst)} evaluations in {time.time() - start:.1f} seconds")

## Step 3: Load once per worker with `block_allocation` and `init_function`
With `block_allocation=True` the executor starts `max_workers` workers, each with the resources defined in the
executor's `resource_dict`, and keeps them alive until the executor is shut down. The `init_function` is executed once
in every worker. It returns a dictionary, and each submitted function receives the entries matching its parameters -
here `potential` - as keyword arguments.

In [ ]:
start = time.time()
with SlurmJobExecutor(
    max_workers=2,
    block_allocation=True,
    init_function=load_potential,
    resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args},
) as exe:
    energy_lst = list(exe.map(evaluate, distance_lst))
print(f"{len(energy_lst)} evaluations in {time.time() - start:.1f} seconds")
energy_lst

The setup is now paid once per worker instead of once per call; with thousands of evaluations the setup cost becomes
negligible. Two restrictions follow from the workers being long-lived:

* All workers share the resources defined for the executor - a `resource_dict` per `submit()` call is not supported.
  Use a second executor for calls with different requirements.
* The `init_function` cannot be combined with a `cache_directory`: the cache key only covers the submitted function
  and its arguments, not the state created by the `init_function`, so executorlib warns about stale results.

## Step 4: Survive crashing workers with `restart_limit`
Compiled libraries occasionally crash with a segmentation fault, which takes down the whole worker process. With
`restart_limit` the executor starts a replacement worker - running the `init_function` again - and resubmits the task
which was running when the worker died. Here the worker is killed while evaluating one of the distances, once:

In [ ]:
import os
import signal

marker_file = os.path.abspath("story6_crash.marker")


def evaluate_unstable(distance, potential):
    # Crash the worker once, like a segmentation fault in a compiled library would.
    if distance == 4.0 and not os.path.exists(marker_file):
        open(marker_file, "w").close()
        os.kill(os.getpid(), signal.SIGKILL)
    return evaluate(distance, potential)


with SlurmJobExecutor(
    max_workers=2,
    block_allocation=True,
    init_function=load_potential,
    restart_limit=2,
    resource_dict={"cores": 1, "slurm_cmd_args": slurm_cmd_args},
) as exe:
    energy_lst = list(exe.map(evaluate_unstable, distance_lst))
energy_lst

Without `restart_limit` (the default is `0`) a crashed worker is not replaced; its tasks are handed to the remaining
workers, and once no worker is left the remaining futures fail with an `ExecutorlibSocketError`.

## Clean up

In [ ]:
if os.path.exists(marker_file):
    os.remove(marker_file)

## What you learned
* `block_allocation=True` keeps `max_workers` workers alive for the lifetime of the executor.
* The `init_function` runs once per worker and provides keyword arguments, such as a loaded model, to every call.
* `restart_limit` replaces crashed workers and resubmits the interrupted task.
* The same parameters are available for the `SingleNodeExecutor` and the `FluxJobExecutor`.

**Reference:** [block allocation](https://executorlib.readthedocs.io/en/latest/1-single-node.html#block-allocation),
[HPC Job Executor](https://executorlib.readthedocs.io/en/latest/3-hpc-job.html#block-allocation) and the API documentation of the
[SlurmJobExecutor](https://executorlib.readthedocs.io/en/latest/api.html).